**1. Визначте DataFrame з тривимірними векторами слів:**

In [208]:
import pandas as pd # Library for Dataframes
import numpy as np # Library for math functions
from sklearn.decomposition import PCA

In [209]:
import pickle # Python object serialization library. Not secure
import requests
import io

url = 'https://drive.google.com/uc?export=download&id=1281E0CDneuKdflWFBUvuyUzujpdGVImz'

# Download the file
response = requests.get(url)

# Load the content into a BytesIO object
with io.BytesIO(response.content) as f:
    word_embeddings = pickle.load(f)

len(word_embeddings) # there should be 243 words that will be used in this assignment

243

In [210]:
#Get the vector for a given word:
def vec(w):
    return word_embeddings[w]

In [211]:
words_all = list(word_embeddings.keys())
all_vectors = np.array([vec(word) for word in words_all])
print(all_vectors.shape)

(243, 300)


In [212]:
words_all.sort()
print(words_all)

['Abuja', 'Accra', 'Afghanistan', 'Albania', 'Algeria', 'Algiers', 'Amman', 'Angola', 'Ankara', 'Antananarivo', 'Apia', 'Armenia', 'Ashgabat', 'Asmara', 'Astana', 'Athens', 'Australia', 'Austria', 'Azerbaijan', 'Baghdad', 'Bahamas', 'Bahrain', 'Baku', 'Bamako', 'Bangkok', 'Bangladesh', 'Banjul', 'Beijing', 'Beirut', 'Belarus', 'Belgium', 'Belgrade', 'Belize', 'Belmopan', 'Berlin', 'Bern', 'Bishkek', 'Botswana', 'Bratislava', 'Brussels', 'Bucharest', 'Budapest', 'Bujumbura', 'Bulgaria', 'Burundi', 'Cairo', 'Canada', 'Canberra', 'Caracas', 'Chile', 'China', 'Chisinau', 'Conakry', 'Copenhagen', 'Croatia', 'Cuba', 'Cyprus', 'Dakar', 'Damascus', 'Denmark', 'Dhaka', 'Doha', 'Dominica', 'Dublin', 'Dushanbe', 'Ecuador', 'Egypt', 'England', 'Eritrea', 'Estonia', 'Fiji', 'Finland', 'France', 'Funafuti', 'Gabon', 'Gaborone', 'Gambia', 'Georgetown', 'Georgia', 'Germany', 'Ghana', 'Greece', 'Greenland', 'Guinea', 'Guyana', 'Hanoi', 'Harare', 'Havana', 'Helsinki', 'Honduras', 'Hungary', 'Indonesia',

In [213]:
from sklearn.decomposition import PCA

# Create axes to plot on
pca = PCA(n_components=3)
embeddings3d = pca.fit_transform(all_vectors)

print(embeddings3d.shape)

(243, 3)


In [214]:
#v = vec(w='Accra')
#print(v)
#print(v.shape)

In [215]:
df_3d = pd.DataFrame(data=embeddings3d, index=words_all)
df_3d['word'] = words_all # додаткова колонка

In [216]:
import plotly.express as px
fig = px.scatter_3d(df_3d, x=0, y=1, z=2, color='word')
fig.show()

**2. Визначте функції для пошуку найближчого слова:**

In [217]:
# words_all: list[str]
# embeddings3d: np.array shape (N, 3)

words_all = np.array(words_all)          # зручно для індексації
E = np.asarray(embeddings3d, dtype=float)  # (N,3)

# Нормалізація для cosine similarity
#E_norm = E / (np.linalg.norm(E, axis=1, keepdims=True) + 1e-12)

In [218]:
def nearest_word(vec3, words=words_all, E=E):
    """
    Find the nearest word using cosine similarity
    WITHOUT pre-normalizing vectors.
    """

    # convert vector to numpy array
    v = np.asarray(vec3, dtype=float).reshape(1, 3)

    # dot product with all embeddings
    dot_products = (E @ v.T).ravel()

    # compute norms
    v_norm = np.linalg.norm(v)
    E_norms = np.linalg.norm(E, axis=1)

    # compute cosine similarity using full formula
    # + 1e-12 - exclude devide by zero error
    sims = dot_products / (E_norms * v_norm + 1e-12)

    # exclude the original word
    top_idx = np.argsort(-sims)[:2]
    # find index of maximum similarity
    #idx = int(np.argmax(sims))
    idx = int(top_idx[1])

    return str(words[idx]), float(sims[idx]), idx

In [219]:
def nearest_words(vec3, top_k=5, words=words_all, E=E):
    """
    Find top_k nearest words using cosine similarity
    WITHOUT pre-normalizing vectors.

    Parameters
    ----------
    vec3 : array-like (3,)
        Input 3D vector.

    top_k : int
        Number of nearest words to return.

    words : array
        Vocabulary list.

    E : ndarray (N,3)
        Matrix of word embeddings.

    Returns
    -------
    list of tuples
        [(word1, similarity1), (word2, similarity2), ...]
    """

    # convert input vector to numpy array
    v = np.asarray(vec3, dtype=float).reshape(1, 3)

    # dot product with all embeddings
    dot_products = (E @ v.T).ravel()

    # compute norms
    v_norm = np.linalg.norm(v)
    E_norms = np.linalg.norm(E, axis=1)

    # full cosine similarity formula
    # + 1e-12 - exclude devide by zero error
    sims = dot_products / (E_norms * v_norm + 1e-12)

    # sort indices by descending similarity
    top_idx = np.argsort(-sims)[1:top_k + 1]

    # return words and similarities
    return [(str(words[i]), float(sims[i])) for i in top_idx]

In [220]:
# Define a function to find the closest word to a vector:
def closest_words(v, top_k=5, exclude_word=None, df=df_3d):
    # Extract numeric embedding matrix (shape: N x 3)
    # We explicitly take only the first 3 columns in case
    # additional non-numeric columns exist (e.g., 'word')
    M = df.iloc[:, :3].values
    # Compute difference between each word vector and input vector
    diff = M - v
    # Get the squared L2 norm of each difference vector.
    # It means the squared euclidean distance from each word to the input vector
    delta = np.sum(diff ** 2, axis=1)
    # If exclude_word is provided,
    # set its distance to infinity so it cannot be selected
    if exclude_word is not None and exclude_word in df.index:
        delta[df.index.get_loc(exclude_word)] = np.inf
    # Get indexes of the smallest distances
    # argsort sorts ascending (smallest distance first)
    idx = np.argsort(delta)[1:top_k + 1]
    # Return list of (word, distance) tuples
    return [(df.index[i], float(delta[i])) for i in idx]

In [221]:
def vec_3d(word):
    # знаходимо індекс слова у words_all
    idx = int(np.where(words_all == word)[0][0])
    if idx == 0:
        return None # Word not found: {word}"
    return E[idx]

In [222]:
# якщо цих слів немає у словнику треба замінити
examples = ["king", "queen", "Kiev", "Moscow"]

In [223]:
for w in examples:
    if w in set(words_all):
        v = vec_3d(w)
        print("-----", w, "-----")
        print("Cosine nearest:", nearest_word(v))
        print("Top5 Сosine nearest:", nearest_words(v))
        print("Top5 L2 nearest:", closest_words(v))
    else:
        print("Missing in vocab:", w)

----- king -----
Cosine nearest: ('Slovakia', 0.9904477440606667, 184)
Top5 Сosine nearest: [('Slovakia', 0.9904477440606667), ('Thailand', 0.983345005820258), ('Zagreb', 0.9708416287875092), ('Norway', 0.9689566529977517), ('Malawi', 0.9666610292015547)]
Top5 L2 nearest: [('Thailand', 0.12776724162472597), ('Norway', 0.21142362588974528), ('Uganda', 0.24472828564269555), ('continent', 0.25361234097924257), ('oil', 0.27624197560128483)]
----- queen -----
Cosine nearest: ('Paramaribo', 0.9689412898116441, 163)
Top5 Сosine nearest: [('Paramaribo', 0.9689412898116441), ('Moldova', 0.9682260306446623), ('Tallinn', 0.9634538790134172), ('Bern', 0.9606605027461547), ('Nepal', 0.9582976005892978)]
Top5 L2 nearest: [('Helsinki', 0.25917800815812664), ('Funafuti', 0.2803465462740995), ('Rwanda', 0.2833741780427479), ('Oman', 0.306175216008886), ('Denmark', 0.37389247127984493)]
----- Kiev -----
Cosine nearest: ('Nassau', 0.9972404542489411, 149)
Top5 Сosine nearest: [('Nassau', 0.99724045424894

In [224]:
random_vec = np.array([0.5, -0.1, 0.2])
print("Random vec ->", nearest_word(random_vec))
print("Top5 Cosine nearest: ->", nearest_words(random_vec))
print("Top5 L2 nearest:", closest_words(random_vec))

Random vec -> ('Ashgabat', 0.9280865780764671, 12)
Top5 Cosine nearest: -> [('Ashgabat', 0.9280865780764671), ('Belarus', 0.9165257863445567), ('Copenhagen', 0.9120431953018132), ('Australia', 0.8969862120337682), ('Armenia', 0.8912968107094383)]
Top5 L2 nearest: [('Ashgabat', 0.12024409167045694), ('Dushanbe', 0.1331311840466654), ('Australia', 0.14827768137256764), ('Armenia', 0.19282034823564487), ('Eritrea', 0.22140552006970707)]


**3. Обчисліть векторний добуток для знаходження ортогонального слова:**



In [225]:
def orthogonal_words_from_pair(word_a, word_b, top_k=5, df=df_3d):
    """
    Compute cross product of 3D vectors of two words and return nearest neighbors
    to the orthogonal vector (excluding the original words).
    """
    a = vec_3d(word_a)
    b = vec_3d(word_b)

    # If any vector is missing, return a safe result
    if a is None or b is None:
        return {
            "pair": (word_a, word_b),
            "status": "missing word(s) in vocabulary",
            "orth_vec_norm": None,
            "neighbors": []
        }

    c = np.cross(a, b)  # orthogonal vector to the plane spanned by a and b
    c_norm = float(np.linalg.norm(c))

    if c_norm < 1e-9:
        return {
            "pair": (word_a, word_b),
            "status": "cross ~ 0 (vectors nearly colinear)",
            "orth_vec": c,
            "orth_vec_norm": c_norm,
            "neighbors": []
        }

    # Find nearest words by L2 distance, exclude the original words
    neighbors = closest_words(c, top_k=top_k+2)  # ask a bit more
    neighbors = [(w, d) for (w, d) in neighbors if w not in {word_a, word_b}][:top_k]
    # Find nearest words by Cosine, exclude the original words
    neighbors_cos = nearest_words(c, top_k=top_k+2)
    neighbors_cos = [(w, sim) for (w, sim) in neighbors_cos if w not in {word_a, word_b}][:top_k]

    return {
        "pair": (word_a, word_b),
        "status": "ok",
        "orth_vec": c,
        "orth_vec_norm": c_norm,
        "neighbors cos": neighbors_cos,
        "neighbors L2": neighbors
    }

In [226]:
pairs = [
    ("king", "queen"),
    ("man", "woman"),
    ("Kiev", "Moscow"),
    ("Paris", "France"),
]

In [227]:
print("Векторні добутки для 3D-векторів:")
for a, b in pairs:
    if a in df_3d.index and b in df_3d.index:
        res = orthogonal_words_from_pair(a, b)
        print(f"\nPair: {a} x {b}")
        print("Status:", res["status"])
        print("Nearest words to cross(a,b) by Cos:")
        for w, sim in res["neighbors cos"]:
            print(f"  {w:15s}  dist={sim:.4f}")
        print("Nearest words to cross(a,b) by L2:")
        for w, d in res["neighbors L2"]:
            print(f"  {w:15s}  dist={d:.4f}")
    else:
        print(f"\nMissing in vocab: {a} or {b}")

Векторні добутки для 3D-векторів:

Pair: king x queen
Status: ok
Nearest words to cross(a,b) by Cos:
  Lisbon           dist=0.9995
  Burundi          dist=0.9871
  village          dist=0.9840
  Honduras         dist=0.9706
  Portugal         dist=0.9699
Nearest words to cross(a,b) by L2:
  Lisbon           dist=0.0873
  Greece           dist=0.4252
  Accra            dist=0.4269
  Honduras         dist=0.4423
  village          dist=0.4601

Missing in vocab: man or woman

Pair: Kiev x Moscow
Status: ok
Nearest words to cross(a,b) by Cos:
  Jakarta          dist=0.9934
  Senegal          dist=0.9908
  Algiers          dist=0.9878
  Dhaka            dist=0.9822
  Bangladesh       dist=0.9806
Nearest words to cross(a,b) by L2:
  Dhaka            dist=0.2392
  Astana           dist=0.2576
  Bangladesh       dist=0.3364
  Bratislava       dist=0.3622
  Algiers          dist=0.3719

Pair: Paris x France
Status: ok
Nearest words to cross(a,b) by Cos:
  Kathmandu        dist=0.9294
  Azerbai

### Пункт 3. Векторний добуток (ортогональне слово)

Для пар слів (A,B) було обчислено ортогональний вектор c = a×b у 3D-просторі (PCA).
Далі знайдено найближчі слова до за L2-відстанню, та Косинусною подібністю.

Спостереження:
- Для деяких пар результат дає слова з частково схожою тематикою знаходить назви міст, але різних країн.
- Для інших пар результати виглядають випадковими, що очікувано, оскільки:
  1) PCA стискає високовимірну семантику до 3 вимірів,
  2) cross product — геометрична операція і не гарантує семантичної інтерпретації,
  3) якщо вектори близькі до колінеарних, a×b ≈ 0 і напрямок нестабільний.

Очікувано що відстань L2 не завжди співпадає з косинусною подібністю.

Cosine similarity → важливий напрямок вектора тому що він кодую зміст

Euclidean distance → важлива реальна геометрична відстань

Після PCA масштаби координат змінюються, тому:
- косинус може показати один найближчий вектор
- евклідова метрика — інший

Висновок: експеримент демонструє, що геометрична ортогональність у 3D PCA не завжди відповідає семантичним відношенням у word embeddings.

**4. Напишіть функції визначення кута між словами:**

In [228]:
def angle_between_words(word_a, word_b):
    # Returns: angle_deg: angle in degrees between vectors of two words

    a = vec_3d(word_a).astype(float)
    b = vec_3d(word_b).astype(float)

    cos_theta = float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

    angle_rad = float(np.arccos(cos_theta))
    angle_deg = float(np.degrees(angle_rad))
    return angle_deg

In [229]:
for a, b in pairs:
    if (a in set(words_all)) and (b in set(words_all)):
        ang = angle_between_words(a, b)
        print(f"{a:10s} vs {b:10s} -> angle = {ang:6.2f}°")
    else:
        print("Missing:", a, "or", b)

king       vs queen      -> angle = 106.81°
Missing: man or woman
Kiev       vs Moscow     -> angle = 108.17°
Paris      vs France     -> angle = 155.33°


## Висновки

Функція angle_between_words() обчислює кут між словами. Малі кути (наприклад, між king і queen) відповідають більшій семантичній близькості, бо ці слова описують людей з певними характеристиками, тоді як великі кути (наприклад, Paris і France) — більшій віддаленості, бо не можна порівнювати місто та країну.

Отримані значення узгоджуються із загальною інтуїцією, але можуть відрізнятися через проєкцію PCA та обмеження підмножини слів.